# F4 career analytics: exploration

Run `make all` first so `data/processed/` exists. This notebook explores the data behind
`reports/results.md`: coverage, how drivers move between series, and individual careers.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / "src"))
import pandas as pd, matplotlib.pyplot as plt
from f4ca.config import Config
cfg = Config()
f4 = pd.read_csv(cfg.processed_dir / "f4_results.csv")
ladder = pd.read_csv(cfg.processed_dir / "ladder.csv")
drivers = pd.read_csv(cfg.processed_dir / "drivers.csv")
coverage = pd.read_csv(cfg.processed_dir / "coverage.csv")
f4.shape, ladder.shape, drivers.shape

## 1. Data coverage
Which season pages were found and parsed? Gaps here limit everything downstream.

In [ ]:
grid = (coverage[coverage.kind == "f4"]
        .pivot_table(index="series", columns="season", values="drivers", aggfunc="sum"))
fig, ax = plt.subplots(figsize=(12, 5))
im = ax.imshow(grid.fillna(0), cmap="Reds", aspect="auto")
ax.set_xticks(range(len(grid.columns)), grid.columns); ax.set_yticks(range(len(grid.index)), grid.index)
for (i, j), v in pd.DataFrame(grid.values).stack().items():
    ax.text(j, i, int(v), ha="center", va="center", fontsize=7)
ax.set_title("Classified drivers per F4 season (blank = page missing or not parsed)", loc="left")
plt.colorbar(im); plt.show()
coverage[coverage.status != "parsed"]

## 2. Data quality checks

In [ ]:
checks = {
    "duplicate driver-season-series rows": f4.duplicated(["driver_id", "series", "season"]).sum(),
    "rows with no race entries": (f4.entries == 0).sum(),
    "drivers matched by name only (no article)": f4.driver_id.str.startswith("name:").mean(),
    "finishes > entries": (f4.finishes > f4.entries).sum(),
}
pd.Series(checks)

## 3. How many seasons do drivers spend in F4?

In [ ]:
drivers.f4_seasons_total.value_counts().sort_index().plot.bar(title="F4 seasons per driver")
plt.show()

## 4. Where do drivers go next?
The most common first higher category for each F4 debut series.

In [ ]:
first_step = (ladder.sort_values("season").merge(drivers[["driver_id", "debut_season", "debut_series"]], on="driver_id")
              .query("season > debut_season").groupby("driver_id").first())
pd.crosstab(first_step.debut_series, first_step.series, margins=True)

## 5. Look up a career

In [ ]:
name = "Antonelli"   # try any surname
mask_f4 = f4.driver.str.contains(name, case=False)
mask_l = ladder.driver.str.contains(name, case=False)
pd.concat([f4[mask_f4].assign(tier="F4"), ladder[mask_l]])[["season", "tier", "series", "driver", "pos", "points", "wins"]].sort_values("season")

## 6. Rookie finish vs outcome, by debut year
Is it getting harder to progress? A falling line at the same finish position would say yes.

In [ ]:
lab = drivers[drivers.labelled]
top3 = lab[lab.best_pos <= 3].groupby("debut_season")["reached_tier3_in_horizon"].mean()
rest = lab[lab.best_pos > 3].groupby("debut_season")["reached_tier3_in_horizon"].mean()
ax = top3.plot(marker="o", label="Rookie top 3"); rest.plot(marker="o", ax=ax, label="Rest")
ax.set_ylabel("Share reaching F3 level in 4 yrs"); ax.legend(); plt.show()